# OCR BNP — Pipeline V10 — 1 appel GPU par page + Export Excel
> `Kernel → Restart Kernel and Run All Cells`


## 1. Dépendances

In [ ]:
%pip install -q pymupdf pillow transformers openpyxl
print('✅ OK')

## 2. Imports

In [ ]:
import time, json, re
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import fitz
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForVision2Seq
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
print('✅ Imports OK')

## 3. Config

In [ ]:
MODEL_PATH      = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen2.5-VL-7B-Instruct/main"
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS  = 700
IMAGE_MAX_SIZE  = 1120
PDF_ZOOM        = 2.0
BLANK_THRESHOLD = 0.97

INPUT_DIR  = Path("/mnt/data/transferts_in")
OUTPUT_DIR = Path("/mnt/data/transferts_out")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXCEL_PATH = OUTPUT_DIR / f"audit_transferts_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

pdfs = sorted(INPUT_DIR.glob("*.pdf"))
print(f'Device    : {DEVICE}')
print(f'Dossiers  : {len(pdfs)}')
print(f'Excel     : {EXCEL_PATH}')

## 4. Chargement modèle

In [ ]:
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16,
    trust_remote_code=True, low_cpu_mem_usage=True,
)
model.eval().to(DEVICE)
print(f'✅ Modèle chargé en {time.time()-t0:.1f}s')

## 5. Utilitaires PDF

In [ ]:
def resize(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side: return img
    r = max_side / max(w, h)
    return img.resize((int(w*r), int(h*r)), Image.LANCZOS)


def is_blank(image, threshold=BLANK_THRESHOLD) -> bool:
    arr = np.array(image.convert('L'))
    return (arr > 240).sum() / arr.size >= threshold


def pdf_to_pages(path: Path, zoom=PDF_ZOOM) -> list:
    doc    = fitz.open(path)
    matrix = fitz.Matrix(zoom, zoom)
    pages  = []
    for i in range(len(doc)):
        pix = doc.load_page(i).get_pixmap(matrix=matrix, alpha=False)
        img = resize(Image.frombytes('RGB', [pix.width, pix.height], pix.samples))
        pages.append({'index': i, 'image': img})
    doc.close()
    return pages


def ask(prompt: str, image: Image.Image) -> str:
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image},
        {'type': 'text',  'text':  prompt},
    ]}]
    text_in = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[text_in], images=[image], return_tensors='pt'
    ).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=1.0,
            top_p=1.0,
            repetition_penalty=1.0,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    generated = out[0][inputs['input_ids'].shape[1]:]
    return processor.decode(generated, skip_special_tokens=True,
                             clean_up_tokenization_spaces=True)


def parse_json(text: str) -> dict:
    try:
        m = re.search(r'\{.*\}', text, re.S)
        return json.loads(m.group()) if m else {}
    except Exception:
        return {}


print('✅ Utilitaires PDF OK')

## 6. Normalisation des données

In [ ]:
def norm_str(v) -> str:
    """Nettoie une chaîne : espaces multiples, strip."""
    if v is None: return None
    s = re.sub(r'\s+', ' ', str(v).strip())
    return s if s and s.lower() not in ('null', 'none', 'n/a') else None


def norm_upper(v) -> str:
    s = norm_str(v)
    return s.upper() if s else None


def norm_compte(v) -> str:
    """Numéro de compte : supprime tous les espaces et caractères non alphanumériques."""
    s = norm_str(v)
    if not s: return None
    return re.sub(r'[^A-Za-z0-9]', '', s).upper()


def norm_montant(v) -> float:
    """Convertit un montant texte en float.
    Gère : '1 261 600.00' / '1,261,600.00' / '1 261 600,00'
    """
    if v is None: return None
    if isinstance(v, (int, float)): return float(v)
    s = str(v).strip()
    s = re.sub(r'[^\d.,]', '', s)        # garder chiffres, point, virgule
    if not s: return None
    # Format français : 1 261 600,00
    if s.count(',') == 1 and '.' not in s:
        s = s.replace(',', '.')
    # Format mixte : 1,261,600.00
    elif '.' in s and ',' in s:
        s = s.replace(',', '')
    # Virgules comme séparateurs milliers : 1,261,600
    elif s.count(',') > 1:
        s = s.replace(',', '')
    try:
        return float(s)
    except:
        return None


def norm_date(v) -> str:
    """Normalise une date au format DD/MM/YYYY si possible."""
    if not v: return None
    s = norm_str(v)
    if not s: return None
    # Déjà au bon format
    if re.match(r'^\d{2}/\d{2}/\d{4}$', s): return s
    # Format avec tirets : 2026-02-24 → 24/02/2026
    m = re.match(r'^(\d{4})-(\d{2})-(\d{2})$', s)
    if m: return f'{m.group(3)}/{m.group(2)}/{m.group(1)}'
    return s


def norm_periode(v) -> str:
    """Normalise la période : JANVIER2026PART1 / JANVIER.2026 → JANVIER 2026 PART 1"""
    if not v: return None
    s = str(v).upper().strip()
    # Remplacer séparateurs par espace
    s = re.sub(r'[._\-]', ' ', s)
    # Coller PART au chiffre : PART1 → PART 1
    s = re.sub(r'PART\s*(\d)', r'PART \1', s)
    # Normaliser espaces
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def normalise_doc(doc_type: str, data: dict) -> dict:
    """Applique les normalisations selon le type de document."""
    d = dict(data)  # copie

    if doc_type == 'OV':
        # ── Aplatissement robuste ────────────────────────────────────────────
        # Le modèle peut retourner les données sous 3 formes :
        #   1. Plat    : {"monnaie": "DZD", ...}
        #   2. Zones   : {"zone_32": {"monnaie": "DZD"}, "zone_50": {...}}
        #   3. Mixte   : {"monnaie": "DZD", "zone_50": {"compte_debiteur": "..."}}
        # On aplatit tout en une passe avant de normaliser.
        for zone_key in list(d.keys()):
            if zone_key.startswith('zone_') and isinstance(d[zone_key], dict):
                d.update(d.pop(zone_key))
            elif isinstance(d.get(zone_key), dict):
                # Sous-dict sans préfixe zone_ → aplatir aussi
                d.update(d.pop(zone_key))

        # Aplatir les sous-dicts de zones si le modèle les a retournés groupés
        # ex: {'zone_32': {'monnaie': 'DZD'}} -> {'monnaie': 'DZD'}
        for zone_key in ['zone_32', 'zone_70', 'zone_50', 'zone_59', 'zone_57']:
            sub = d.pop(zone_key, None)
            if isinstance(sub, dict):
                d.update(sub)

        d['monnaie']             = norm_upper(d.get('monnaie'))
        d['montant_chiffres']    = norm_montant(d.get('montant_chiffres'))
        d['montant_lettres']     = norm_str(d.get('montant_lettres'))
        d['periode']             = norm_periode(d.get('periode'))
        d['tranche']             = norm_str(d.get('tranche'))
        d['date_demande']        = norm_date(d.get('date_demande'))
        d['compte_debiteur']     = norm_compte(d.get('compte_debiteur'))
        d['beneficiaire_nom']    = norm_upper(d.get('beneficiaire_nom'))
        d['beneficiaire_compte'] = norm_compte(d.get('beneficiaire_compte'))
        d['beneficiaire_adresse']= norm_str(d.get('beneficiaire_adresse'))
        d['swift']               = norm_compte(d.get('swift'))  # sans espaces
        d['banque_nom']          = norm_upper(d.get('banque_nom'))

    elif doc_type == 'ANNEXE_II':
        d['mois_transfert']    = norm_periode(d.get('mois_transfert'))
        d['nom_prenom']        = norm_upper(d.get('nom_prenom'))
        d['compte_bancaire']   = norm_compte(d.get('compte_bancaire'))
        d['salaire_mensuel']   = norm_montant(d.get('salaire_mensuel'))
        d['nombre_jours']      = norm_str(d.get('nombre_jours'))
        d['part_transferable'] = norm_montant(d.get('part_transferable'))
        d['pays_destination']  = norm_upper(d.get('pays_destination'))
        d['compte_devise']     = norm_str(d.get('compte_devise'))

    elif doc_type == 'ANNEXE_I':
        d['nom_prenom']      = norm_upper(d.get('nom_prenom'))
        d['date_signature']  = norm_date(d.get('date_signature'))

    elif doc_type == 'BULLETIN':
        d['nom_prenom']       = norm_upper(d.get('nom_prenom'))
        d['matricule']        = norm_str(d.get('matricule'))
        d['mois_bulletin']    = norm_periode(d.get('mois_bulletin'))
        d['salaire_base']     = norm_montant(d.get('salaire_base'))
        d['salaire_brut']     = norm_montant(d.get('salaire_brut'))
        d['retenue_ss']       = norm_montant(d.get('retenue_ss'))
        d['retenue_irg']      = norm_montant(d.get('retenue_irg'))
        d['retenue_mutuelle'] = norm_montant(d.get('retenue_mutuelle'))
        d['net_a_payer']      = norm_montant(d.get('net_a_payer'))

    return d


print('✅ Normalisation OK')

## 7. Prompt universel

In [ ]:
PROMPT_UNIVERSEL = """\
Lis ce document bancaire.

ÉTAPE 1 — Identifie le type en lisant le titre principal :
- OV        : titre "ORDRE DE VIREMENT A L'ETRANGER"
- ANNEXE_I  : titre "Annexe I"
- ANNEXE_II : titre "Annexe II"
- BULLETIN  : titre "BULLETIN DE PAIE"
- AUTRE     : tout autre document (page vide, tableau, cachet...)

ÉTAPE 2 — Selon le type identifié, extrais les champs ci-dessous.
Si le type est AUTRE, retourne uniquement {"type": "AUTRE"}.

--- Si OV ---
Zone 32 : monnaie, montant_chiffres, montant_lettres
Zone 70 : periode (mois + année bruts ex: JANVIER 2026), tranche (Part 1 / Part 2 / null)
Zone 50 : date_demande, compte_debiteur (20 chiffres commençant par 02700)
Zone 59 : beneficiaire_nom, beneficiaire_compte, beneficiaire_adresse
Zone 57 : swift, banque_nom

--- Si ANNEXE_II ---
mois_transfert (valeur brute après "Mois de"), nom_prenom,
compte_bancaire, salaire_mensuel, nombre_jours (null si absent),
part_transferable (montant chiffres uniquement),
pays_destination, compte_devise (banque + numéro tels qu'écrits)

--- Si ANNEXE_I ---
nom_prenom, date_signature

--- Si BULLETIN ---
nom_prenom, matricule, mois_bulletin,
salaire_base, salaire_brut, retenue_ss, retenue_irg, retenue_mutuelle,
net_a_payer

RÈGLES :
- Retourne UNIQUEMENT un JSON valide, sans texte avant ni après
- Commence toujours par {"type": "..."}
- Si un champ est absent ou illisible : null
- Pas de markdown, pas de backticks
"""

print(f'✅ Prompt : {len(PROMPT_UNIVERSEL)} caractères')

## 8. Traitement d'un PDF

In [ ]:
def process_pdf(pdf_path: Path, verbose=True) -> dict:
    """
    Traite toutes les pages d'un PDF.
    Retourne {
      'fichier'          : nom du fichier,
      'date_traitement'  : horodatage,
      'pages_trouvees'   : liste des types détectés,
      'pages_manquantes' : types attendus mais absents,
      'anomalies'        : doublons détectés,
      'OV'               : {...},
      'ANNEXE_I'         : {...},
      'ANNEXE_II'        : {...},
      'BULLETIN'         : {...},
    }
    """
    TYPES_ATTENDUS = {'OV', 'ANNEXE_I', 'ANNEXE_II', 'BULLETIN'}

    pages    = pdf_to_pages(pdf_path)
    results  = {}
    doublons = []

    if verbose:
        print(f'\n📁 {pdf_path.name} — {len(pages)} page(s)')

    for page in pages:
        i   = page['index']
        img = page['image']

        # Page blanche → ignorer sans appel GPU
        if is_blank(img):
            if verbose: print(f'  Page {i+1} → ignorée (vide)')
            continue

        # 1 seul appel GPU
        t0       = time.time()
        raw      = ask(PROMPT_UNIVERSEL, img)
        elapsed  = time.time() - t0
        data     = parse_json(raw)
        doc_type = data.get('type', 'INCONNU')

        if doc_type in ('AUTRE', 'INCONNU'):
            if verbose: print(f'  Page {i+1} → {doc_type:10s} | {elapsed:.1f}s — ignorée')
            continue

        # Normalisation des données
        data = normalise_doc(doc_type, data)

        if verbose:
            print(f'  Page {i+1} → {doc_type:10s} | {elapsed:.1f}s')
            for k, v in data.items():
                if k != 'type' and v is not None:
                    print(f'    {k:25s}: {v}')

        # Doublon ?
        if doc_type in results:
            doublons.append(doc_type)
            if verbose: print(f'    ⚠️  DOUBLON {doc_type} — ignoré')
            continue

        results[doc_type] = data

    # Résumé
    pages_trouvees   = sorted(results.keys())
    pages_manquantes = sorted(TYPES_ATTENDUS - set(results.keys()))

    return {
        'fichier':          pdf_path.name,
        'date_traitement':  datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'pages_trouvees':   ', '.join(pages_trouvees),
        'pages_manquantes': ', '.join(pages_manquantes) if pages_manquantes else None,
        'anomalies':        'DOUBLON: ' + ', '.join(doublons) if doublons else None,
        **{t: results.get(t, {}) for t in TYPES_ATTENDUS},
    }


print('✅ process_pdf OK')

## 9. Export Excel

In [ ]:
# ── Schéma des colonnes par document ─────────────────────────────────────────
SCHEMA = {
    'META': [
        ('fichier',          'Fichier'),
        ('date_traitement',  'Date traitement'),
        ('pages_trouvees',   'Pages trouvées'),
        ('pages_manquantes', 'Pages manquantes'),
        ('anomalies',        'Anomalies'),
    ],
    'OV': [
        ('monnaie',              'Monnaie'),
        ('montant_chiffres',     'Montant (chiffres)'),
        ('montant_lettres',      'Montant (lettres)'),
        ('periode',              'Période'),
        ('tranche',              'Tranche'),
        ('date_demande',         'Date demande'),
        ('compte_debiteur',      'Compte débiteur'),
        ('beneficiaire_nom',     'Bénéficiaire nom'),
        ('beneficiaire_compte',  'Bénéficiaire compte'),
        ('beneficiaire_adresse', 'Bénéficiaire adresse'),
        ('swift',                'SWIFT'),
        ('banque_nom',           'Banque bénéficiaire'),
    ],
    'ANNEXE_I': [
        ('nom_prenom',     'Nom Prénom'),
        ('date_signature', 'Date signature'),
    ],
    'ANNEXE_II': [
        ('mois_transfert',    'Mois transfert'),
        ('nom_prenom',        'Nom Prénom'),
        ('compte_bancaire',   'Compte bancaire'),
        ('salaire_mensuel',   'Salaire mensuel'),
        ('nombre_jours',      'Nombre de jours'),
        ('part_transferable', 'Part transférable'),
        ('pays_destination',  'Pays destination'),
        ('compte_devise',     'Compte devise'),
    ],
    'BULLETIN': [
        ('nom_prenom',       'Nom Prénom'),
        ('matricule',        'Matricule'),
        ('mois_bulletin',    'Mois bulletin'),
        ('salaire_base',     'Salaire base'),
        ('salaire_brut',     'Salaire brut'),
        ('retenue_ss',       'Retenue SS'),
        ('retenue_irg',      'Retenue IRG'),
        ('retenue_mutuelle', 'Retenue mutuelle'),
        ('net_a_payer',      'Net à payer'),
    ],
}

# Couleurs
COLORS = {
    'META':      {'header': 'FF1F4E79', 'col': 'FFD6E4F0'},
    'OV':        {'header': 'FF833C00', 'col': 'FFFCE4D6'},
    'ANNEXE_I':  {'header': 'FF375623', 'col': 'FFE2EFDA'},
    'ANNEXE_II': {'header': 'FF203864', 'col': 'FFDAE3F3'},
    'BULLETIN':  {'header': 'FF3F3151', 'col': 'FFEDE7F6'},
}


def create_excel(path: Path, rows: list):
    """Crée le fichier Excel avec en-têtes colorés et toutes les données."""
    wb = Workbook()
    ws = wb.active
    ws.title = 'Dossiers'

    # ── Construction de la liste ordonnée des colonnes ────────────────────────
    all_cols = []   # (groupe, field_key, label)
    for groupe, cols in SCHEMA.items():
        for field_key, label in cols:
            all_cols.append((groupe, field_key, label))

    # ── Ligne 1 : groupes fusionnés colorés ───────────────────────────────────
    col_idx   = 1
    group_map = defaultdict(list)
    for groupe, _, _ in all_cols:
        group_map[groupe].append(col_idx)
        col_idx += 1

    for groupe, cols in group_map.items():
        s, e = cols[0], cols[-1]
        if s < e:
            ws.merge_cells(start_row=1, start_column=s, end_row=1, end_column=e)
        c = ws.cell(row=1, column=s)
        c.value     = groupe
        c.font      = Font(bold=True, color='FFFFFFFF', name='Arial', size=11)
        c.fill      = PatternFill('solid', start_color=COLORS[groupe]['header'])
        c.alignment = Alignment(horizontal='center', vertical='center')
    ws.row_dimensions[1].height = 22

    # ── Ligne 2 : noms des colonnes ───────────────────────────────────────────
    for i, (groupe, _, label) in enumerate(all_cols, start=1):
        c = ws.cell(row=2, column=i)
        c.value     = label
        c.font      = Font(bold=True, name='Arial', size=9)
        c.fill      = PatternFill('solid', start_color=COLORS[groupe]['col'])
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        c.border    = Border(bottom=Side(style='thin'), right=Side(style='hair'))
        ws.column_dimensions[get_column_letter(i)].width = 20
    ws.row_dimensions[2].height = 35
    ws.freeze_panes = ws.cell(row=3, column=len(SCHEMA['META']) + 1)

    # ── Lignes de données ─────────────────────────────────────────────────────
    for row_num, dossier in enumerate(rows, start=3):
        for col_idx, (groupe, field_key, _) in enumerate(all_cols, start=1):
            if groupe == 'META':
                val = dossier.get(field_key)
            else:
                val = dossier.get(groupe, {}).get(field_key)

            c = ws.cell(row=row_num, column=col_idx)
            c.value = val
            c.font  = Font(name='Arial', size=9)
            c.fill  = PatternFill('solid', start_color=COLORS[groupe]['col'])

            # Format nombre pour les montants
            if isinstance(val, float):
                c.number_format = '0.00'

            # Alerte rouge si anomalie
            if groupe == 'META' and field_key == 'anomalies' and val:
                c.font = Font(name='Arial', size=9, bold=True, color='FFCC0000')
            if groupe == 'META' and field_key == 'pages_manquantes' and val:
                c.font = Font(name='Arial', size=9, bold=True, color='FFCC0000')

    wb.save(path)
    print(f'✅ Excel sauvegardé : {path}')
    print(f'   {len(rows)} dossier(s) | {len(all_cols)} colonnes')


print('✅ Export Excel OK')

## 10. Pipeline complet

In [ ]:
SEP = '=' * 72
rows    = []
t_total = time.time()

for pdf_path in pdfs:
    print(SEP)
    try:
        result = process_pdf(pdf_path, verbose=True)
        rows.append(result)
        manq = result.get('pages_manquantes')
        anom = result.get('anomalies')
        print(f'  ✅ Docs trouvés  : {result["pages_trouvees"]}')
        if manq: print(f'  ⚠️  Manquants    : {manq}')
        if anom: print(f'  🔴 Anomalies    : {anom}')
    except Exception as e:
        print(f'  ❌ ERREUR : {e}')

# Export Excel
print(f'\n{SEP}')
create_excel(EXCEL_PATH, rows)
elapsed = time.time() - t_total
print(f'⏱  Temps total : {elapsed:.1f}s ({elapsed/max(1,len(rows)):.1f}s/dossier)')
print(f'📊 {len(rows)} dossiers traités')

## 11. Analyse rapide des résultats

In [ ]:
import pandas as pd

df = pd.read_excel(EXCEL_PATH, header=1)
print(f'📊 {len(df)} dossiers')
print()

# Dossiers avec anomalies
col_anom = 'Anomalies'
if col_anom in df.columns:
    anom = df[df[col_anom].notna()]
    print(f'🔴 Anomalies (doublons) : {len(anom)}')
    if len(anom): print(anom[['Fichier', col_anom]].to_string(index=False))
    print()

# Dossiers incomplets
col_manq = 'Pages manquantes'
if col_manq in df.columns:
    manq = df[df[col_manq].notna()]
    print(f'⚠️  Dossiers incomplets : {len(manq)}')
    if len(manq): print(manq[['Fichier', col_manq]].to_string(index=False))
    print()

display(df.head(5))